In [1]:
#confusion_matrix_dtop_weight_modedu = {0:-1}
#params_dtop_weight_modedu = {0:-1}

%store -r

In [2]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, fbeta_score, recall_score, precision_score
#from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
#from pprint import pprint
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score


df = pd.read_csv("term-deposit-marketing-2020.csv")

df.replace("unknown", np.nan, inplace=True)

df["y"] = df["y"].map({"no": 0, "yes": 1})

binary_cols = ["housing", "loan", "default"]

for col in binary_cols:
    df[col] = df[col].map({"no": 0, "yes": 1})

#df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})
df = df.drop(columns="contact")
df.replace("unknown", np.nan, inplace=True)

/data/Apziva/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,age,job,marital,education,default,balance,housing,loan,day,month,duration,campaign,y
0,58,management,married,tertiary,0,2143,1,0,5,may,261,1,0
1,44,technician,single,secondary,0,29,1,0,5,may,151,1,0
2,33,entrepreneur,married,secondary,0,2,1,1,5,may,76,1,0
3,47,blue-collar,married,NaN,0,1506,1,0,5,may,92,1,0
4,33,NaN,single,NaN,0,1,0,0,5,may,198,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,53,technician,married,tertiary,0,395,0,0,3,jun,107,1,0
39996,30,management,single,tertiary,0,3340,0,0,3,jun,238,3,1
39997,54,admin,divorced,secondary,0,200,0,0,3,jun,170,1,1
39998,34,management,married,tertiary,0,1047,0,0,3,jun,342,1,0


In [3]:
known_education = df[df["education"].notna()].copy()
unknown_education = df[df["education"].isna()].copy()

In [4]:
education_features = ['age', 'job', 'marital', 'default', 'balance', 'housing', 'loan']
X_education = known_education[education_features]
#X_education = known_education.drop(["education", "y", "month"], axis=1)
y_education = known_education["education"]
X_missing = unknown_education[education_features]

In [5]:
drop_cols = [
    "duration",
    "campaign",
    "month",
    "day"
]

df = df.drop(columns=drop_cols)

In [6]:
X_education = pd.get_dummies(
    X_education,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [7]:
X_missing = pd.get_dummies(
    X_missing,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_education,
    y_education,
    test_size=0.2,
    random_state=1234,
    stratify=y_education
)

In [9]:
education_model = DecisionTreeClassifier(
    random_state=1234,
    max_depth=10,
    min_samples_leaf=5,
    class_weight={'primary': 2, 'secondary': 1, 'tertiary': 1}
)

In [10]:
education_model.fit(X_train, y_train)
education_test_pred = education_model.predict(X_test)

In [11]:
print(accuracy_score(y_test, education_test_pred))

0.6757213413049129


In [12]:
y_education.value_counts(normalize=True)

education
secondary    0.545712
tertiary     0.291299
primary      0.162988
Name: proportion, dtype: float64

In [13]:
print(confusion_matrix(y_test, education_test_pred))

[[ 816  371   67]
 [ 961 2961  277]
 [ 138  681 1422]]


In [14]:
education_predictions = education_model.predict(X_missing)

df.loc[unknown_education.index, "education"] = education_predictions

In [15]:
df["education"].isna().sum()

np.int64(0)

In [16]:
df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})

In [17]:
df = pd.get_dummies(
    df,
    columns=["job", "marital"],
    drop_first=True,
    dtype=int
)

In [18]:
#df.info()

In [19]:
#raise Exception('pause for checking.')

In [20]:
X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1234,
    stratify=y
)

In [21]:
##!
##!
#beta2 = {'four':2.0, 'qrtr':0.5, 'one':1.0, 'half':0.707, 'two':1.414, 'three':1.73, 'third':0.577, 'zero':0.0, 'inf':10000, 'ten':3.16,
#        'five':2.23, '4.5':2.12, '4.25':2.06}

#strvar = 'one'
#weight_list = ['balanced', None, {0:1, 1:2}, {0:1, 1:3}, {0:1, 1:4}, {0:1, 1:5}, {0:1, 1:6}]

beta_scorer = make_scorer(fbeta_score, beta=1, pos_label=1)
#beta_scorer = make_scorer(recall_score, pos_label=1)


In [22]:
cw_map = {
      1: None, #bad
      0: "balanced", #better
      2: {0: 1, 1: 2}, #not great
      3: {0: 1, 1: 3}, #not great
      4: {0: 1, 1: 4}, #not great
      5: {0: 1, 1: 5}, #bad
      6: {0: 1, 1: 6},
      7: {0: 1, 1: 7},
      8: {0: 1, 1: 8},
      9: {0: 1, 1: 9},
      10: {0: 1, 1: 10},
      11: {0: 1, 1: 11},
      12: {0: 1, 1: 12},
      13: {0: 1, 1: 13},
      14: {0: 1, 1: 14},
      20: {0: 1, 1: 20}, #great
      21: {0: 1, 1: 21},
      16: {0: 1, 1: 16},
      15: {0: 1, 1: 15},
      22: {0: 1, 1: 22},
      23: {0: 1, 1: 23},
      24: {0: 1, 1: 24},
      25: {0: 1, 1: 25},
      26: {0: 1, 1: 26},
      27: {0: 1, 1: 27},
      28: {0: 1, 1: 28},
      29: {0: 1, 1: 29},
      30: {0: 1, 1: 30},
      19: {0: 1, 1: 19},
      18: {0: 1, 1: 18},
      17: {0: 1, 1: 17},
      31: {0: 1, 1: 31},
}

In [23]:
num = 17  #not done 3-15
def objective(trial):

    cw_opt = trial.suggest_categorical("class_weight_opt", [num])#[0, 1, 2, 3, 4, 5])

    model = DecisionTreeClassifier(
        ccp_alpha=trial.suggest_float('ccp_alpha',0.0, 0.02),
        max_depth=trial.suggest_int("max_depth", 2, 30), #3-15 / 2-30
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20), #2-20 /
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 15), #1-10 /1-20
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        #class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),#,{0:1, 1:2}, {0:1, 1:3}]),#, {0:1, 1:4}, {0:2, 1:1}]),
        class_weight=cw_map[cw_opt],
        splitter=trial.suggest_categorical('splitter', ['best', 'random']),
        criterion=trial.suggest_categorical("criterion", ['gini', 'entropy', 'log_loss']),
        #class_weight=['balanced'],
        random_state=1234
        
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=2,
        scoring=beta_scorer
    )

    return scores.mean()

In [24]:
sampler = optuna.samplers.TPESampler(seed=1234)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=sampler)

In [25]:
study.optimize(objective, n_trials=250) #had 250

In [26]:
print(study.best_params)
print(study.best_value)

{'class_weight_opt': 17, 'ccp_alpha': 0.0007872784421100068, 'max_depth': 16, 'min_samples_split': 5, 'min_samples_leaf': 14, 'max_features': None, 'splitter': 'random', 'criterion': 'gini'}
0.15799872124408493


In [27]:
best = study.best_params
cw_opt = best.pop("class_weight_opt")

best_model = DecisionTreeClassifier(
    random_state=1234,
    class_weight=cw_map[cw_opt],
    #**study.best_params
    **best
)

best_model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'random'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",14
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",1234
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curr

In [28]:
#best_model.tree_.node_count

In [29]:
#print(best_model.get_depth())
#print(best_model.get_n_leaves())

In [30]:
#print(best_model.tree_.weighted_n_node_samples)

In [31]:
#print(best_model.class_weight)

In [32]:
pred = best_model.predict(X_test)

In [33]:
print("this is the confustion matrix:\n", confusion_matrix(y_test, pred))
print("classification report:\n", classification_report(y_test, pred))
print("this is the weight", cw_map[num])

this is the confustion matrix:
 [[2326 5095]
 [  90  489]]
classification report:
               precision    recall  f1-score   support

           0       0.96      0.31      0.47      7421
           1       0.09      0.84      0.16       579

    accuracy                           0.35      8000
   macro avg       0.53      0.58      0.32      8000
weighted avg       0.90      0.35      0.45      8000

this is the weight {0: 1, 1: 17}


In [34]:
confusion_matrix_dtop_weight_modedu[num]=confusion_matrix(y_test, pred)

params_dtop_weight_modedu[num]=study.best_params

for weight, matrix in sorted(confusion_matrix_dtop_weight_modedu.items()):
    print(f"{weight}:\n {matrix}")

print("\n")

for weight, params in sorted(params_dtop_weight_modedu.items()):
    print(f"{weight}:\n {params}")

print("\n")

%store confusion_matrix_dtop_weight_modedu
%store params_dtop_weight_modedu

0:
 [[4499 2922]
 [ 240  339]]
1:
 [[7134  287]
 [ 539   40]]
17:
 [[2326 5095]
 [  90  489]]
29:
 [[1639 5782]
 [  83  496]]
30:
 [[2919 4502]
 [ 162  417]]
31:
 [[2679 4742]
 [ 140  439]]


0:
 {'class_weight_opt': 0, 'ccp_alpha': 0.00045423479105262583, 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 2, 'max_features': None, 'splitter': 'best', 'criterion': 'entropy'}
1:
 {'class_weight_opt': 1, 'ccp_alpha': 3.84314306986248e-05, 'max_depth': 28, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'log2', 'splitter': 'best', 'criterion': 'entropy'}
17:
 {'class_weight_opt': 17, 'ccp_alpha': 0.0007872784421100068, 'max_depth': 16, 'min_samples_split': 5, 'min_samples_leaf': 14, 'max_features': None, 'splitter': 'random', 'criterion': 'gini'}
29:
 {'class_weight_opt': 29, 'ccp_alpha': 1.6476827000452138e-05, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 15, 'max_features': 'log2', 'splitter': 'best', 'criterion': 'gini'}
30:
 {'class_weight_opt':

In [35]:
params_dt_fcw_op = study.best_params
#weighting=beta2_params_dt_fcw_op[strvar]['class_weight_opt']
#print(weighting)
#beta2_params_dt_fcw_op[strvar]['class_weight']=cw_map[weighting]
matrix_dt_fcw_op = confusion_matrix(y_test, pred)

In [36]:
#for key in beta2_params_dt_fcw_op:
print(params_dt_fcw_op)

{'class_weight_opt': 17, 'ccp_alpha': 0.0007872784421100068, 'max_depth': 16, 'min_samples_split': 5, 'min_samples_leaf': 14, 'max_features': None, 'splitter': 'random', 'criterion': 'gini'}


In [37]:
for row in matrix_dt_fcw_op:
    print(row)#for element in row:

[2326 5095]
[ 90 489]


In [38]:
#raise Exception('stopping for now')

In [39]:
precision, recall, fbeta, support = precision_recall_fscore_support(
    y_test,
    pred,
    beta=1
)

for i in range(len(precision)):
#    print(f"beta value: {beta2[strvar]}")
    print(f"Class {i}")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  Fbeta:   {fbeta[i]:.4f}")
    print(f"  Support:   {support[i]}")

accuracy = accuracy_score(y_test, pred)

print(f"Accuracy: {accuracy:.4f}")

Class 0
  Precision: 0.9627
  Recall:    0.3134
  Fbeta:   0.4729
  Support:   7421
Class 1
  Precision: 0.0876
  Recall:    0.8446
  Fbeta:   0.1587
  Support:   579
Accuracy: 0.3519


In [40]:
#precision, recall, fbeta, support = precision_recall_fscore_support(
#    y_test,
#    pred,
#    beta=beta2[strvar]
#)
#beta2_scores_dt_fcw_op[strvar] = {'precision0':-1}
#for i in range(len(precision)):
#    #print(f"beta value: {beta2[strvar]}")
#    #print(f"Class {i}")
#    #print(f"  Precision: {precision[i]:.4f}")
#    #if i == 0: 
#        #beta2_scores_dt_fcw_op[strvar]['precision0'] = precision[i]; beta2_scores_dt_fcw_op[strvar]['recall0'] = recall[i]; 
#        #beta2_scores_dt_fcw_op[strvar]['fbeta0'] = fbeta[i]
#    #if i == 1: 
#        #beta2_scores_dt_fcw_op[strvar]['precision1'] = precision[i]; beta2_scores_dt_fcw_op[strvar]['recall1'] = recall[i];
#        #beta2_scores_dt_fcw_op[strvar]['fbeta1'] = fbeta[i]
#    #print(f"  Recall:    {recall[i]:.4f}")
#    #print(f"  Fbeta:   {fbeta[i]:.4f}")
#    #print(f"  Support:   {support[i]}")

accuracy = accuracy_score(y_test, pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.3519


In [41]:
for label in [0, 1]:
    score = fbeta_score(
        y_test,
        pred,
        beta=1,
        pos_label=label
    )
    print(f"Class {label} Fbeta-score: {score:.4f}")

print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
#print(f"dict of dict: ", beta2_scores_dt_fcw_op)

Class 0 Fbeta-score: 0.4729
Class 1 Fbeta-score: 0.1587
Accuracy: 0.3519
